# 1D Lennard-Jones MD for argon, in JAX

NVE molecular dynamics of identical, uncharged particles on a periodic line, written with `jax` (autodiff forces, `jit`, `lax.scan`). The physics lives in `lj.py`; this notebook runs it and checks the results against theory.

Units: length in Å, energy in K (energy / k_B), mass in u, time in the unit that follows from these.

| Section | Question |
|---|---|
| 1 | Do autodiff and analytic forces agree? |
| 2 | Does energy minimisation reach r = 2^(1/6) σ from any start? |
| 3 | Does the bound-pair dynamics follow the 1/\|v\| distribution? |
| 4 | Does the small-oscillation frequency match sqrt(U''/μ)? |
| 5 | Does the energy drift scale as dt²? |
| 6 | How does g(r) of a 1D chain compare with exp(−U/T)? |

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import lj

P = lj.ARGON
SIGMA, EPS, MASS = P["sigma"], P["epsilon"], P["mass"]
R_MIN = 2 ** (1 / 6) * SIGMA
DT = 0.002  # time step
BOX_PAIR = 20 * SIGMA  # large box: a single pair does not feel its own image

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Forces: autodiff vs analytic

Forces come from `-jax.grad` of one energy function. As a check, compare them with the hand-derived LJ force.

In [ ]:
r = jnp.linspace(0.9 * SIGMA, 3.0 * SIGMA, 400)
f_ana = lj.lj_force_analytic(r, P)
f_auto = jax.vmap(lambda ri: lj.forces(jnp.array([0.0, ri]), BOX_PAIR, P)[1])(r)
err = jnp.abs(f_auto - f_ana) / jnp.max(jnp.abs(f_ana))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ax1.plot(r / SIGMA, lj.pair_energy(r, P) / EPS, label="U(r)")
ax1.plot(r / SIGMA, f_ana * SIGMA / EPS, label="F(r), analytic")
ax1.plot(r / SIGMA, f_auto * SIGMA / EPS, "--", label="F(r), autodiff")
ax1.axvline(2 ** (1 / 6), color="grey", lw=0.8)
ax1.set(xlabel="r / σ", ylabel="U / ε,  F σ / ε", ylim=(-2, 4))
ax1.legend()
ax2.semilogy(r / SIGMA, err + 1e-18)
ax2.set(xlabel="r / σ", ylabel="|F_auto − F_analytic| / max|F|", title="float64 agreement")
fig.tight_layout()
plt.show()

## 2. Two particles: energy minimisation

Hypothesis: steepest descent ends at the same separation r = 2^(1/6) σ ≈ 3.82 Å for any start. (NVE dynamics cannot do this: it conserves energy and oscillates forever.)

In [ ]:
starts = jnp.linspace(0.92, 2.5, 15) * SIGMA
finals, iters = [], []
for r0 in starts:
    pos, n_iter = lj.minimize_energy(jnp.array([0.0, r0]), BOX_PAIR, P)
    finals.append(pos[1] - pos[0])
    iters.append(int(n_iter))
finals = jnp.array(finals)

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(starts / SIGMA, finals / SIGMA, "o", label="minimised separation")
ax.axhline(2 ** (1 / 6), color="C3", lw=1, label=r"$2^{1/6}$")
ax.set(xlabel="start separation / σ", ylabel="final separation / σ")
ax.legend()
fig.tight_layout()
plt.show()
print(f"max |r_final - r_min| = {float(jnp.max(jnp.abs(finals - R_MIN))):.2e} Å;  iterations: {min(iters)}-{max(iters)}")

## 3. Two particles: dynamics

Start at rest at r₀ = 1.3 σ, so the energy E = U(r₀) is negative and the pair is bound. The relative motion has reduced mass μ = m/2, and the time spent at separation r is proportional to 1/|ṙ| = 1/sqrt(2 (E − U(r)) / μ), with spikes at the turning points.

In [ ]:
r0 = 1.3 * SIGMA
state = lj.init_state(jnp.array([0.0, r0]), BOX_PAIR, P)
n_steps = 40_000
_, traj = lj.run(state, BOX_PAIR, DT, n_steps, P)
sep = traj.pos[:, 1] - traj.pos[:, 0]
t = DT * jnp.arange(1, n_steps + 1)
E0 = float(traj.total[0])

grid = jnp.linspace(0.95 * SIGMA, 3.0 * SIGMA, 6000)
p_pred = lj.predicted_density(grid, E0, P)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
m = t < 14
ax1.plot(t[m], sep[m] / SIGMA)
ax1.axhline(2 ** (1 / 6), color="grey", lw=0.8)
ax1.set(xlabel="t", ylabel="r / σ", title="separation")
counts, _, _ = ax2.hist(np.asarray(sep / SIGMA), bins=40, density=True, alpha=0.6, label="MD time histogram")
ax2.plot(grid / SIGMA, p_pred * SIGMA, "C3", label=r"prediction $\propto 1/|\dot r|$")
ax2.axvline(2 ** (1 / 6), color="grey", lw=0.8)
ax2.set(xlabel="r / σ", ylabel="density", xlim=(0.95, 1.4), ylim=(0, 1.5 * counts.max()), title="where the pair spends its time")
ax2.legend()
fig.tight_layout()
plt.show()
print(f"E = {E0 / EPS:.3f} eps,  mean r = {float(sep.mean()) / SIGMA:.3f} sigma (r_min = {2 ** (1 / 6):.3f} sigma)")

## 4. Small oscillations: frequency

For small amplitude the well is harmonic: ω = sqrt(U''(r_min) / μ). U'' is taken with `jax.grad` twice. Larger amplitudes probe the soft outer wall, so the period grows.

In [ ]:
k = jax.grad(jax.grad(lj.pair_energy))(R_MIN, P)  # U''(r_min)
period0 = 2 * jnp.pi / jnp.sqrt(k / (MASS / 2))

amps = jnp.array([0.005, 0.01, 0.02, 0.05, 0.1, 0.2])  # relative displacement from r_min
periods = []
for a in amps:
    state = lj.init_state(jnp.array([0.0, R_MIN * (1 + a)]), BOX_PAIR, P)
    _, tr = lj.run(state, BOX_PAIR, DT, 8000, P)
    s = tr.pos[:, 1] - tr.pos[:, 0]
    s = s - s.mean()
    up = jnp.where((s[:-1] < 0) & (s[1:] >= 0))[0]  # upward zero crossings
    t_cross = (up + (-s[up] / (s[up + 1] - s[up]))) * DT
    periods.append(float(jnp.mean(jnp.diff(t_cross))))

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.semilogx(amps, jnp.array(periods), "o-", label="measured")
ax.axhline(float(period0), color="C3", label=r"harmonic: $2\pi\sqrt{\mu/U''}$")
ax.set(xlabel="amplitude / r_min", ylabel="period")
ax.legend()
fig.tight_layout()
plt.show()
print(f"U''(r_min) = {float(k * SIGMA**2 / EPS):.2f} eps/sigma^2   (analytic 36 * 2^(2/3) = {36 * 2 ** (2 / 3):.2f})")

## 5. Energy conservation vs time step

Velocity Verlet is second order, so the maximum relative energy error should scale as dt².

In [ ]:
state = lj.init_state(jnp.array([0.0, 1.3 * SIGMA]), BOX_PAIR, P)
T_TOTAL = 4.0  # total time, a few oscillation periods
dts = np.array([0.0005, 0.001, 0.002, 0.004, 0.008])
drifts = []
for dt in dts:
    _, tr = lj.run(state, BOX_PAIR, float(dt), int(round(T_TOTAL / dt)), P)
    drifts.append(float(jnp.max(jnp.abs(tr.total - tr.total[0])) / abs(tr.total[0])))
slope = np.polyfit(np.log(dts), np.log(drifts), 1)[0]

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.loglog(dts, drifts, "o-", label=f"measured, slope {slope:.2f}")
ax.loglog(dts, drifts[0] * (dts / dts[0]) ** 2, "k--", lw=1, label=r"$\propto dt^2$")
ax.set(xlabel="dt", ylabel="max |ΔE| / |E|")
ax.legend()
fig.tight_layout()
plt.show()

## 6. A chain of N particles

N particles on a periodic line, started on a lattice with a random displacement and zero velocity. The displacement stores potential energy, so the total energy is set at the start and the temperature is *measured* (equipartition) over the second half of the run. The first half is discarded as equilibration.

Hypothesis: for a dilute system g(r) ≈ exp(−U(r) / T). A dense chain is solid-like, with sharp peaks at multiples of the spacing, so the low-density formula should fail there.

Caveat: particles in 1D cannot pass each other and thermalise slowly, so g(r) is only roughly converged.

In [ ]:
N = 40

def run_chain(rho_star, jitter, n_steps=100_000, seed=0):
    """NVE chain of N particles at reduced density rho* = N * sigma / box."""
    box = N * SIGMA / rho_star
    pos = lj.init_lattice(N, box, jitter, jax.random.PRNGKey(seed))
    state = lj.init_state(pos, box, P)
    _, traj = lj.run(state, box, DT, n_steps, P)
    return box, traj

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (rho, jit, name) in zip(axes, [(0.5, 0.3, "dilute"), (0.9, 0.1, "dense")]):
    box, traj = run_chain(rho, jit)
    half = traj.pos.shape[0] // 2
    T = float(jnp.mean(2 * traj.kinetic[half:] / (N - 1)))
    r, g = lj.g_of_r(jnp.mod(traj.pos[half::20], box), box, n_bins=120, r_max=4 * SIGMA)
    ax.plot(r / SIGMA, g, label="MD")
    ax.plot(r / SIGMA, lj.boltzmann_gr(r, T, P), "--", label=r"$e^{-U/T}$")
    ax.set(xlabel="r / σ", title=f"{name}: ρ* = {rho}, T = {T / EPS:.2f} ε / k_B ({T:.0f} K)", ylim=(0, 4))
    ax.legend()
axes[0].set_ylabel("g(r)")
fig.tight_layout()
plt.show()